In [1]:
#============
#CELL NO-: 01
#IoT-23 files are Zeek's native conn.log format, not CSV.
#They have comment lines starting with # (header metadata),
#tab-separated data rows, and a #close footer line at the end.
#Let's look at the raw structure of one file first before writing
#a parser.
#============

FILE_PATH = r"F:\c2-iot2\data\raw\iot23_samples\Capture-8-1_conn.log.labeled"

with open(FILE_PATH, 'r') as f:
    raw_lines = f.readlines()

print("Total lines in file:", len(raw_lines))
print("\n--- First 10 lines (raw) ---")
for line in raw_lines[:10]:
    print(repr(line))

print("\n--- Last 2 lines (raw) ---")
for line in raw_lines[-2:]:
    print(repr(line))

Total lines in file: 10412

--- First 10 lines (raw) ---
'#separator \\x09\n'
'#set_separator\t,\n'
'#empty_field\t(empty)\n'
'#unset_field\t-\n'
'#path\tconn\n'
'#open\t2018-08-08-12-22-16\n'
'#fields\tts\tuid\tid.orig_h\tid.orig_p\tid.resp_h\tid.resp_p\tproto\tservice\tduration\torig_bytes\tresp_bytes\tconn_state\tlocal_orig\tlocal_resp\tmissed_bytes\thistory\torig_pkts\torig_ip_bytes\tresp_pkts\tresp_ip_bytes\ttunnel_parents   label   detailed-label\n'
'#types\ttime\tstring\taddr\tport\taddr\tport\tenum\tstring\tinterval\tcount\tcount\tstring\tbool\tbool\tcount\tstring\tcount\tcount\tcount\tcount\tset[string]   string   string\n'
'1533042911.474174\tC5JLGOoxIw2dBZt47\t192.168.100.113\t123\t81.2.254.224\t123\tudp\t-\t0.005490\t48\t48\tSF\t-\t-\t0\tDd\t1\t76\t1\t76\t(empty)   Benign   -\n'
'1533042921.474234\tCf3cHf4jZr9nvD808i\t192.168.100.113\t123\t147.231.100.5\t123\tudp\t-\t0.001741\t48\t48\tSF\t-\t-\t0\tDd\t1\t76\t1\t76\t(empty)   Benign   -\n'

--- Last 2 lines (raw) ---
'153312

In [2]:
#============
#CELL NO-: 02
#Now we write a function that turns the raw Zeek log lines into a
#clean list of rows. Steps:
#1. Skip any line starting with # (these are header/footer metadata)
#2. Split each data line by tab - this gives us 20 fields
#3. The 20th field actually contains 3 values squashed together with
#   spaces (tunnel_parents, label, detailed-label) - we split that
#   separately using regex
#============

import re

# These are the 20 tab-separated field names, in order, from the #fields line
zeek_columns = [
    'ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
    'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
    'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
    'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes'
]

def parse_zeek_log(file_path):
    """
    Reads one IoT-23 Zeek conn.log.labeled file and returns a list of
    dictionaries, one per data row (header/footer lines are skipped).
    """
    rows = []
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue  # skip header/footer metadata lines
            if not line.strip():
                continue  # skip any blank lines

            parts = line.rstrip('\n').split('\t')

            # first 20 parts map directly to zeek_columns
            row = dict(zip(zeek_columns, parts[:20]))

            # the 21st part (index 20) contains "tunnel_parents label detailed-label"
            # separated by spaces, not tabs - use regex to split it into 3 pieces
            last_field = parts[20] if len(parts) > 20 else ""
            match = re.match(r'^(\S+)\s+(\S+)\s+(\S+)$', last_field)
            if match:
                row['tunnel_parents'] = match.group(1)
                row['label'] = match.group(2)
                row['detailed_label'] = match.group(3)
            else:
                # if the pattern doesn't match, mark it so we can inspect later
                row['tunnel_parents'] = None
                row['label'] = None
                row['detailed_label'] = None

            rows.append(row)
    return rows

# Test the parser on Capture-8-1 first
test_rows = parse_zeek_log(FILE_PATH)
print(f"Parsed {len(test_rows)} rows from Capture-8-1")
print(f"\nFirst row: {test_rows[0]}")
print(f"\nHow many rows failed to parse the label (label is None): {sum(1 for r in test_rows if r['label'] is None)}")

Parsed 10403 rows from Capture-8-1

First row: {'ts': '1533042911.474174', 'uid': 'C5JLGOoxIw2dBZt47', 'id.orig_h': '192.168.100.113', 'id.orig_p': '123', 'id.resp_h': '81.2.254.224', 'id.resp_p': '123', 'proto': 'udp', 'service': '-', 'duration': '0.005490', 'orig_bytes': '48', 'resp_bytes': '48', 'conn_state': 'SF', 'local_orig': '-', 'local_resp': '-', 'missed_bytes': '0', 'history': 'Dd', 'orig_pkts': '1', 'orig_ip_bytes': '76', 'resp_pkts': '1', 'resp_ip_bytes': '76', 'tunnel_parents': '(empty)', 'label': 'Benign', 'detailed_label': '-'}

How many rows failed to parse the label (label is None): 0


In [3]:
#============
#CELL NO-: 03
#Now we loop through all 5 capture files, parse each one, and add a
#'source_capture' column so we know which file each row came from.
#Then we combine everything into a single dataframe.
#We combine all 5 captures instead of using just one, because a single
#capture (like Capture-8-1) is 79% one attack type - using all 5 gives
#more malware family diversity and a bigger, more reliable test set.
#============

import pandas as pd

RAW_DIR = r"F:\c2-iot2\data\raw\iot23_samples"

capture_files = [
    "Capture-8-1_conn.log.labeled",
    "Capture-20-1_conn.log.labeled",
    "Capture-21-1_conn.log.labeled",
    "Capture-42-1_conn.log.labeled",
    "Capture-44-1_conn.log.labeled",
]

all_rows = []

for filename in capture_files:
    full_path = RAW_DIR + "\\" + filename
    capture_name = filename.split("_conn.log.labeled")[0]  # e.g. "Capture-8-1"

    parsed_rows = parse_zeek_log(full_path)

    # tag every row with which capture file it came from
    for row in parsed_rows:
        row['source_capture'] = capture_name

    all_rows.extend(parsed_rows)
    print(f"{capture_name}: {len(parsed_rows)} rows parsed")

# Combine everything into one dataframe
iot23_df = pd.DataFrame(all_rows)

print(f"\nTotal combined rows: {iot23_df.shape[0]}")
print(f"Total columns: {iot23_df.shape[1]}")
print(f"\nRows per capture:")
print(iot23_df['source_capture'].value_counts())

Capture-8-1: 10403 rows parsed
Capture-20-1: 3209 rows parsed
Capture-21-1: 3286 rows parsed
Capture-42-1: 4426 rows parsed
Capture-44-1: 237 rows parsed

Total combined rows: 21561
Total columns: 24

Rows per capture:
source_capture
Capture-8-1     10403
Capture-42-1     4426
Capture-21-1     3286
Capture-20-1     3209
Capture-44-1      237
Name: count, dtype: int64


In [4]:
#============
#CELL NO-: 04
#Two things to fix here:
#1. label is currently text ("Benign"/"Malicious") - we map it to
#   0/1 to match the TON_IoT label format (0=normal, 1=attack)
#2. IoT-23 uses TWO different missing-value markers: "-" and "(empty)"
#   We convert both to actual NaN so pandas treats them as missing
#============

# Map label: Benign -> 0, Malicious -> 1 (same meaning as TON_IoT's label column)
label_map = {'Benign': 0, 'Malicious': 1}
iot23_df['label'] = iot23_df['label'].map(label_map)

print("Label value counts after mapping:")
print(iot23_df['label'].value_counts(dropna=False))

# Replace both missing-value markers with actual NaN across the whole dataframe
iot23_df = iot23_df.replace(['-', '(empty)'], pd.NA)

print("\nMissing values per column (top 10):")
print(iot23_df.isna().sum().sort_values(ascending=False).head(10))

Label value counts after mapping:
label
0    13277
1     8284
Name: count, dtype: int64

Missing values per column (top 10):
local_orig        21561
tunnel_parents    21561
local_resp        21561
service           17318
detailed_label    13277
duration           9304
orig_bytes         9304
resp_bytes         9304
history               1
proto                 0
dtype: int64


In [5]:
#============
#CELL NO-: 05
#IoT-23 (Zeek native) and TON_IoT (processed) use different column
#names for the same underlying measurement. Before we can compare
#schemas or use a common model, we need to rename IoT-23 columns to
#match TON_IoT's naming.
#Example: orig_bytes (Zeek) = src_bytes (TON_IoT) - both mean
#"bytes sent by the connection originator"
#============

zeek_to_ton_iot_rename = {
    'duration': 'duration',           # same name already
    'orig_bytes': 'src_bytes',
    'resp_bytes': 'dst_bytes',
    'conn_state': 'conn_state',       # same name already
    'missed_bytes': 'missed_bytes',   # same name already
    'orig_pkts': 'src_pkts',
    'orig_ip_bytes': 'src_ip_bytes',
    'resp_pkts': 'dst_pkts',
    'resp_ip_bytes': 'dst_ip_bytes',
    'proto': 'proto',                 # same name already
    'service': 'service',             # same name already
}

iot23_df = iot23_df.rename(columns=zeek_to_ton_iot_rename)

print("IoT-23 dataframe columns after rename:")
print(list(iot23_df.columns))

IoT-23 dataframe columns after rename:
['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'tunnel_parents', 'label', 'detailed_label', 'source_capture']


In [6]:
#============
#CELL NO-: 06
#Here we ACTUALLY DERIVE the Common-11 column list, instead of
#hardcoding it. We load the Set B column list that Notebook 02 saved
#to feature_sets.json (TON_IoT's real, confirmed schema), and take
#the intersection with IoT-23's actual loaded columns (after renaming).
#This makes the 11-column list a defensible, reproducible result,
#not an assumption from memory.
#============

import json

FEATURE_SETS_PATH = r"F:\c2-iot2\data\processed\feature_sets.json"
with open(FEATURE_SETS_PATH, "r") as f:
    feature_sets = json.load(f)

ton_iot_set_b_cols = feature_sets["set_b_columns"]

print(f"TON_IoT Set B columns ({len(ton_iot_set_b_cols)}): {ton_iot_set_b_cols}")
print(f"\nIoT-23 columns ({len(iot23_df.columns)}): {list(iot23_df.columns)}")

# The actual intersection: which Set B columns genuinely exist in IoT-23?
common_cols = [c for c in ton_iot_set_b_cols if c in iot23_df.columns]

print(f"\n--- DERIVED Common columns ---")
print(f"Common columns: {common_cols}")
print(f"Count: {len(common_cols)}")

# Which Set B columns are TON_IoT-specific and NOT available in IoT-23?
ton_iot_only = [c for c in ton_iot_set_b_cols if c not in iot23_df.columns]
print(f"\nSet B columns NOT available in IoT-23 (TON_IoT-specific): {ton_iot_only}")

TON_IoT Set B columns (21): ['proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'http_request_body_len', 'http_response_body_len', 'http_status_code']

IoT-23 columns (24): ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'tunnel_parents', 'label', 'detailed_label', 'source_capture']

--- DERIVED Common columns ---
Common columns: ['proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes']
Count: 11

Set B columns NOT available in IoT-23 (TON_IoT-specific): ['dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dn

In [7]:
#============
#CELL NO-: 07
#Right now, every column in iot23_df is text (string), because we
#parsed everything from a raw text file. The Common-11 numeric
#columns need to be converted to actual numbers before a model can
#use them. pd.to_numeric with errors='coerce' turns any value that
#can't be converted (like NaN we set earlier) into a proper NaN.
#============

numeric_common_cols = [
    'duration', 'src_bytes', 'dst_bytes', 'missed_bytes',
    'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes'
]

for col in numeric_common_cols:
    iot23_df[col] = pd.to_numeric(iot23_df[col], errors='coerce')

print("Data types after conversion:")
print(iot23_df[numeric_common_cols].dtypes)

print("\nQuick sanity check - summary stats:")
print(iot23_df[numeric_common_cols].describe())

Data types after conversion:
duration        float64
src_bytes       float64
dst_bytes       float64
missed_bytes      int64
src_pkts          int64
src_ip_bytes      int64
dst_pkts          int64
dst_ip_bytes      int64
dtype: object

Quick sanity check - summary stats:
           duration     src_bytes      dst_bytes  missed_bytes      src_pkts  \
count  12257.000000  1.225700e+04   12257.000000  21561.000000  2.156100e+04   
mean      18.517484  1.424034e+05     230.477360      6.211725  6.387259e+01   
std      763.560614  1.576018e+07    4652.048303    261.214828  8.888286e+03   
min        0.000313  0.000000e+00       0.000000      0.000000  0.000000e+00   
25%        0.001747  2.900000e+01       0.000000      0.000000  1.000000e+00   
50%        0.008242  4.800000e+01      45.000000      0.000000  1.000000e+00   
75%        3.100236  4.800000e+01      48.000000      0.000000  1.000000e+00   
max    44325.615625  1.744830e+09  171878.000000  20272.000000  1.305034e+06   

       

In [8]:
#============
#CELL NO-: 08
#Save two files:
#1. The full processed IoT-23 dataframe (all columns, for reference)
#2. A Common-11-only subset (the derived common columns from Cell 06)
#   + label + source_capture, ready to feed into the model for the
#   T3-B test
#============

# 1. Full processed dataframe (for reference / future use)
FULL_IOT23_PATH = r"F:\c2-iot2\data\processed\iot23_processed_full.csv"
iot23_df.to_csv(FULL_IOT23_PATH, index=False)
print(f"Saved full IoT-23 dataframe to: {FULL_IOT23_PATH}")
print(f"Shape: {iot23_df.shape}")

# 2. Common-11 subset, ready for the T3-B test
#    (features + label as target + source_capture for per-capture reporting)
iot23_common11 = iot23_df[common_cols + ['label', 'source_capture']]
COMMON11_IOT23_PATH = r"F:\c2-iot2\data\processed\iot23_common11_for_t3b.csv"
iot23_common11.to_csv(COMMON11_IOT23_PATH, index=False)
print(f"\nSaved Common-11 IoT-23 subset to: {COMMON11_IOT23_PATH}")
print(f"Shape: {iot23_common11.shape}")

print("\nFinal check - label distribution in the Common-11 subset:")
print(iot23_common11['label'].value_counts())

Saved full IoT-23 dataframe to: F:\c2-iot2\data\processed\iot23_processed_full.csv
Shape: (21561, 24)

Saved Common-11 IoT-23 subset to: F:\c2-iot2\data\processed\iot23_common11_for_t3b.csv
Shape: (21561, 13)

Final check - label distribution in the Common-11 subset:
label
0    13277
1     8284
Name: count, dtype: int64


In [9]:
#============
#CELL NO-: 09
#The Common-11 list was derived here (Cell 06), by comparing against
#IoT-23's real schema - but TON_IoT also needs its own Common-11 CSV
#file, for the T3-A test (Common-11 model evaluated on a TON_IoT
#holdout, before being transferred to IoT-23 for T3-B).
#We load TON_IoT's full processed dataframe (saved in Notebook 02)
#and select the same derived common_cols from it.
#============

import pandas as pd
import json

TON_IOT_PROCESSED_PATH = r"F:\c2-iot2\data\processed\train_test_network_processed.csv"
ton_iot_df = pd.read_csv(TON_IOT_PROCESSED_PATH)

ton_iot_common11 = ton_iot_df[common_cols + ['label', 'type']]
TON_IOT_COMMON11_PATH = r"F:\c2-iot2\data\processed\common11_features.csv"
ton_iot_common11.to_csv(TON_IOT_COMMON11_PATH, index=False)

print(f"Saved TON_IoT Common-11 subset to: {TON_IOT_COMMON11_PATH}")
print(f"Shape: {ton_iot_common11.shape}")

# Update feature_sets.json so it now records the derived Common-11 list too
FEATURE_SETS_PATH = r"F:\c2-iot2\data\processed\feature_sets.json"
with open(FEATURE_SETS_PATH, "r") as f:
    feature_sets = json.load(f)

feature_sets["common11_columns"] = common_cols
with open(FEATURE_SETS_PATH, "w") as f:
    json.dump(feature_sets, f, indent=2)

print(f"\nUpdated feature_sets.json with common11_columns: {common_cols}")

Saved TON_IoT Common-11 subset to: F:\c2-iot2\data\processed\common11_features.csv
Shape: (190474, 13)

Updated feature_sets.json with common11_columns: ['proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes']
